In [ ]:
!pip install evaluate
!pip install -U transformers datasets accelerate imbalanced-learn

[SKIPPED: pip-only]


In [2]:
import warnings
warnings.filterwarnings("ignore")
import gc
import numpy as np
import pandas as pd
import itertools
from collections import Counter
import matplotlib.pyplot as plt
from sklearn.metrics import accuracy_score, roc_auc_score, confusion_matrix, classification_report, f1_score
from imblearn.over_sampling import RandomOverSampler
import accelerate
import evaluate
from datasets import Dataset, Image, ClassLabel
from transformers import TrainingArguments, Trainer, ViTImageProcessor, ViTForImageClassification, DefaultDataCollator
import torch
from torch.utils.data import DataLoader
from torchvision.transforms import CenterCrop, Compose, Normalize, RandomRotation, RandomResizedCrop, RandomHorizontalFlip, RandomAdjustSharpness, Resize, ToTensor
from PIL import ImageFile
ImageFile.LOAD_TRUNCATED_IMAGES = True
from pathlib import Path
from tqdm import tqdm
import os

In [3]:
file_names = []
labels = []

for file in sorted((Path('/kaggle/input/datasets/manjilkarki/deepfake-and-real-images/Dataset/').glob('*/*/*.*'))):
    label = str(file).split('/')[-2]
    labels.append(label)
    file_names.append(str(file))

print(len(file_names), len(labels))
df = pd.DataFrame.from_dict({"image": file_names, "label": labels})
print(df.shape)
print(df['label'].unique())

190335 190335
CPU subsample to 2000 images
(2000, 2)
<ArrowStringArray>
['Fake', 'Real']
Length: 2, dtype: str


In [4]:
y = df[['label']]
df = df.drop(['label'], axis=1)
ros = RandomOverSampler(random_state=83)
df, y_resampled = ros.fit_resample(df, y)
del y
df['label'] = y_resampled
del y_resampled
gc.collect()
print(df.shape)

(2042, 2)


In [5]:
dataset = Dataset.from_pandas(df).cast_column("image", Image())

labels_list = ['Real', 'Fake']
label2id, id2label = dict(), dict()
for i, label in enumerate(labels_list):
    label2id[label] = i
    id2label[i] = label

print("Mapping of IDs to Labels:", id2label)
print("Mapping of Labels to IDs:", label2id)

ClassLabels = ClassLabel(num_classes=len(labels_list), names=labels_list)

def map_label2id(example):
    example['label'] = ClassLabels.str2int(example['label'])
    return example

dataset = dataset.map(map_label2id, batched=True)
dataset = dataset.cast_column('label', ClassLabels)
dataset = dataset.train_test_split(test_size=0.4, shuffle=True, stratify_by_column="label")
train_data = dataset['train']
test_data = dataset['test']
print("Train size:", len(train_data))
print("Test size:", len(test_data))

Mapping of IDs to Labels: {0: 'Real', 1: 'Fake'}
Mapping of Labels to IDs: {'Real': 0, 'Fake': 1}
Casting the dataset: 100%|##########| 2042/2042 [00:00<00:00, 407303.06 examples/s]
Train size: 1225
Test size: 817


In [6]:
model_str = "dima806/deepfake_vs_real_image_detection"
processor = ViTImageProcessor.from_pretrained(model_str)
image_mean, image_std = processor.image_mean, processor.image_std
size = processor.size["height"]
print("Size:", size)

normalize = Normalize(mean=image_mean, std=image_std)

_train_transforms = Compose([
    Resize((size, size)),
    RandomRotation(90),
    RandomAdjustSharpness(2),
    ToTensor(),
    normalize
])

_val_transforms = Compose([
    Resize((size, size)),
    ToTensor(),
    normalize
])

def train_transforms(examples):
    examples['pixel_values'] = [_train_transforms(image.convert("RGB")) for image in examples['image']]
    return examples

def val_transforms(examples):
    examples['pixel_values'] = [_val_transforms(image.convert("RGB")) for image in examples['image']]
    return examples

train_data.set_transform(train_transforms)
test_data.set_transform(val_transforms)

def collate_fn(examples):
    pixel_values = torch.stack([example["pixel_values"] for example in examples])
    labels = torch.tensor([example['label'] for example in examples])
    return {"pixel_values": pixel_values, "labels": labels}

Size: 224


In [7]:
model = ViTForImageClassification.from_pretrained(model_str, num_labels=len(labels_list))
model.config.id2label = id2label
model.config.label2id = label2id
print("Trainable parameters (millions):", model.num_parameters(only_trainable=True) / 1e6)

Loading weights: 100%|##########| 200/200 [00:00<00:00, 7969.04it/s]
Trainable parameters (millions): 85.800194


In [8]:
accuracy = evaluate.load("accuracy")

def compute_metrics(eval_pred):
    predictions = eval_pred.predictions
    label_ids = eval_pred.label_ids
    predicted_labels = predictions.argmax(axis=1)
    acc_score = accuracy.compute(predictions=predicted_labels, references=label_ids)['accuracy']
    return {"accuracy": acc_score}

model_name = "deepfake_vs_real_image_detection"

args = TrainingArguments(
    output_dir=model_name,
    eval_strategy="epoch",
    learning_rate=1e-6,
    per_device_train_batch_size=32,
    per_device_eval_batch_size=8,
    num_train_epochs=2,
    weight_decay=0.02,
    warmup_steps=50,
    remove_unused_columns=False,
    save_strategy='epoch',
    load_best_model_at_end=True,
    save_total_limit=1,
    report_to="none"
)

trainer = Trainer(
    model=model,
    args=args,
    train_dataset=train_data,
    eval_dataset=test_data,
    data_collator=collate_fn,
    compute_metrics=compute_metrics,
    processing_class=processor,
)

trainer.train()

100%|#########9| 204/205 [03:07<00:00,  1.06it/s]
                                                 
{'eval_loss': '0.01801', 'eval_accuracy': '0.9939', 'eval_runtime': '188.7', 'eval_samples_per_second': '4.329', 'eval_steps_per_second': '1.086', 'epoch': '1'}
100%|##########| 205/205 [03:07<00:00,  1.38it/s]
                                                 
Writing model shards: 100%|##########| 1/1 [00:00<00:00,  4.95it/s]
{'train_runtime': '984.3', 'train_samples_per_second': '1.245', 'train_steps_per_second': '0.312', 'train_loss': '0.03769', 'epoch': '1'}
100%|##########| 307/307 [16:24<00:00,  3.21s/it]


In [9]:
trainer.evaluate()
outputs = trainer.predict(test_data)
print(outputs.metrics)

y_true = outputs.label_ids
y_pred = outputs.predictions.argmax(1)

accuracy_val = accuracy_score(y_true, y_pred)
f1 = f1_score(y_true, y_pred, average='macro')
print(f"Accuracy: {accuracy_val:.4f}")
print(f"F1 Score: {f1:.4f}")

cm = confusion_matrix(y_true, y_pred)

plt.figure(figsize=(8, 6))
plt.imshow(cm, interpolation='nearest', cmap=plt.cm.Blues)
plt.title('Confusion Matrix')
plt.colorbar()
tick_marks = np.arange(len(labels_list))
plt.xticks(tick_marks, labels_list, rotation=90)
plt.yticks(tick_marks, labels_list)
thresh = cm.max() / 2.0
for i, j in itertools.product(range(cm.shape[0]), range(cm.shape[1])):
    plt.text(j, i, format(cm[i, j], '.0f'), horizontalalignment="center", color="white" if cm[i, j] > thresh else "black")
plt.ylabel('True label')
plt.xlabel('Predicted label')
plt.tight_layout()
plt.show()

print("\nClassification report:")
print(classification_report(y_true, y_pred, target_names=labels_list, digits=4))

100%|##########| 205/205 [02:52<00:00,  1.19it/s]
{'test_loss': 0.018014587461948395, 'test_accuracy': 0.9938800489596084, 'test_runtime': 173.3077, 'test_samples_per_second': 4.714, 'test_steps_per_second': 1.183}
Accuracy: 0.9939
F1 Score: 0.9939

Classification report:
              precision    recall  f1-score   support

        Real     0.9951    0.9927    0.9939       409
        Fake     0.9927    0.9951    0.9939       408

    accuracy                         0.9939       817
   macro avg     0.9939    0.9939    0.9939       817
weighted avg     0.9939    0.9939    0.9939       817



In [ ]:
!pip install evaluate
!pip install -U transformers datasets accelerate imbalanced-learn

[SKIPPED: pip-only]


In [11]:
import warnings
warnings.filterwarnings("ignore")
import gc
import numpy as np
import pandas as pd
import itertools
import matplotlib.pyplot as plt
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report, f1_score
from imblearn.over_sampling import RandomOverSampler
import evaluate
from datasets import Dataset, Image, ClassLabel
from transformers import TrainingArguments, Trainer, ViTImageProcessor, ViTForImageClassification
import torch
from torchvision.transforms import Compose, Normalize, RandomRotation, RandomAdjustSharpness, Resize, ToTensor
from PIL import ImageFile
ImageFile.LOAD_TRUNCATED_IMAGES = True
from pathlib import Path
import os
print("Imports done!")

Imports done!


In [12]:
import os
print("=== YOUR MODEL ===")
for dirname, _, filenames in os.walk('/kaggle/input/deepfake-detector'):
    for f in filenames:
        print(os.path.join(dirname, f))

print("\n=== OPENFAKE DATASET ===")
for dirname, _, filenames in os.walk('/kaggle/input'):
    for f in filenames[:2]:
        print(os.path.join(dirname, f))
    if len(filenames) > 2:
        print(f"  ...and {len(filenames)-2} more files")

=== YOUR MODEL ===

=== OPENFAKE DATASET ===
C:/Users/USER/Downloads/kaggle_run/input\datasets\manjilkarki\deepfake-and-real-images\Dataset\Test\Fake\fake_0.jpg
C:/Users/USER/Downloads/kaggle_run/input\datasets\manjilkarki\deepfake-and-real-images\Dataset\Test\Fake\fake_1.jpg
  ...and 5490 more files
C:/Users/USER/Downloads/kaggle_run/input\datasets\manjilkarki\deepfake-and-real-images\Dataset\Test\Real\real_0.jpg
C:/Users/USER/Downloads/kaggle_run/input\datasets\manjilkarki\deepfake-and-real-images\Dataset\Test\Real\real_1.jpg
  ...and 5411 more files
C:/Users/USER/Downloads/kaggle_run/input\datasets\manjilkarki\deepfake-and-real-images\Dataset\Train\Fake\fake_0.jpg
C:/Users/USER/Downloads/kaggle_run/input\datasets\manjilkarki\deepfake-and-real-images\Dataset\Train\Fake\fake_1.jpg
  ...and 69999 more files
C:/Users/USER/Downloads/kaggle_run/input\datasets\manjilkarki\deepfake-and-real-images\Dataset\Train\Real\real_0.jpg
C:/Users/USER/Downloads/kaggle_run/input\datasets\manjilkarki\de

In [ ]:
from transformers import pipeline
from PIL import Image
from IPython.display import display
import ipywidgets as widgets
import io

model_str = "/kaggle/input/models/ayush3102kumar/deepfake-detector/transformers/default/1"
pipe = pipeline('image-classification', model=model_str, device=0)
print("Model loaded! Now upload an image:")

uploader = widgets.FileUpload(accept='image/*', multiple=False)
output = widgets.Output()

def on_upload(change):
    with output:
        output.clear_output()
        uploaded_file = uploader.value[0]  # new API — it's a tuple now
        image = Image.open(io.BytesIO(uploaded_file['content'])).convert("RGB")
        display(image)
        result = pipe(image)
        label = result[0]['label']
        confidence = result[0]['score'] * 100
        print(f"\nLabel: {label}")
        print(f"Confidence: {confidence:.2f}%")
        if 'fake' in label.lower():
            print("🔴 DEEPFAKE DETECTED")
        else:
            print("🟢 REAL IMAGE")

uploader.observe(on_upload, names='value')
display(uploader, output)

[SKIPPED: interactive upload widget]


In [14]:
file_names = []
labels = []

for file in sorted(Path('/kaggle/input/datasets/sanketghadge1/openfake-data-20k-img/openfake_dataset/').glob('*/*/*.*')):
    label = str(file).split('/')[-2]
    if label == 'fake':
        labels.append('Fake')
        file_names.append(str(file))
    elif label == 'real':
        labels.append('Real')
        file_names.append(str(file))

print(f"Total images: {len(file_names)}")
df = pd.DataFrame.from_dict({"image": file_names, "label": labels})
print(df['label'].value_counts())

Total images before subsample: 0
Total images: 0
Series([], Name: count, dtype: int64)


In [ ]:
y = df[['label']]
df = df.drop(['label'], axis=1)
ros = RandomOverSampler(random_state=83)
df, y_resampled = ros.fit_resample(df, y)
del y
df['label'] = y_resampled
del y_resampled
gc.collect()
print(f"Balanced shape: {df.shape}")
print(df['label'].value_counts())

ValueError: Unknown label type: unknown. Maybe you are trying to fit a classifier, which expects discrete classes on a regression target with continuous values.

In [ ]:
from datasets import Dataset, ClassLabel
from datasets import Image as HFImage

dataset = Dataset.from_pandas(df).cast_column("image", HFImage())

labels_list = ['Real', 'Fake']
label2id, id2label = dict(), dict()
for i, label in enumerate(labels_list):
    label2id[label] = i
    id2label[i] = label

print("ID to Label:", id2label)

ClassLabels = ClassLabel(num_classes=len(labels_list), names=labels_list)

def map_label2id(example):
    example['label'] = ClassLabels.str2int(example['label'])
    return example

dataset = dataset.map(map_label2id, batched=True)
dataset = dataset.cast_column('label', ClassLabels)
dataset = dataset.train_test_split(test_size=0.2, shuffle=True, stratify_by_column="label")
train_data = dataset['train']
test_data = dataset['test']
print(f"Train: {len(train_data)}, Test: {len(test_data)}")

ArrowNotImplementedError: Unsupported cast from double to struct using function cast_struct

In [17]:
model_str = "/kaggle/input/models/ayush3102kumar/deepfake-detector/transformers/default/1"

processor = ViTImageProcessor.from_pretrained(model_str)
image_mean, image_std = processor.image_mean, processor.image_std
size = processor.size["height"]
print(f"Image size: {size}")

normalize = Normalize(mean=image_mean, std=image_std)

_train_transforms = Compose([
    Resize((size, size)),
    RandomRotation(90),
    RandomAdjustSharpness(2),
    ToTensor(),
    normalize
])

_val_transforms = Compose([
    Resize((size, size)),
    ToTensor(),
    normalize
])

def train_transforms(examples):
    examples['pixel_values'] = [_train_transforms(image.convert("RGB")) for image in examples['image']]
    return examples

def val_transforms(examples):
    examples['pixel_values'] = [_val_transforms(image.convert("RGB")) for image in examples['image']]
    return examples

train_data.set_transform(train_transforms)
test_data.set_transform(val_transforms)

def collate_fn(examples):
    pixel_values = torch.stack([example["pixel_values"] for example in examples])
    labels = torch.tensor([example['label'] for example in examples])
    return {"pixel_values": pixel_values, "labels": labels}

print("Transforms ready!")

Image size: 224
Transforms ready!


In [18]:
model = ViTForImageClassification.from_pretrained(
    model_str,
    num_labels=len(labels_list),
    ignore_mismatched_sizes=True
)
model.config.id2label = id2label
model.config.label2id = label2id
print(f"Trainable parameters: {model.num_parameters(only_trainable=True)/1e6:.1f}M")

Loading weights: 100%|##########| 200/200 [00:00<00:00, 2452.30it/s]
Trainable parameters: 85.8M


In [19]:
accuracy_metric = evaluate.load("accuracy")

def compute_metrics(eval_pred):
    predictions = eval_pred.predictions
    label_ids = eval_pred.label_ids
    predicted_labels = predictions.argmax(axis=1)
    acc_score = accuracy_metric.compute(predictions=predicted_labels, references=label_ids)['accuracy']
    return {"accuracy": acc_score}

args = TrainingArguments(
    output_dir="deepfake_vs_real_v2",
    eval_strategy="epoch",
    learning_rate=5e-7,
    per_device_train_batch_size=32,
    per_device_eval_batch_size=8,
    num_train_epochs=2,
    weight_decay=0.02,
    warmup_steps=50,
    remove_unused_columns=False,
    save_strategy='epoch',
    load_best_model_at_end=True,
    save_total_limit=1,
    report_to="none"
)

trainer = Trainer(
    model=model,
    args=args,
    train_dataset=train_data,
    eval_dataset=test_data,
    data_collator=collate_fn,
    compute_metrics=compute_metrics,
    processing_class=processor,
)

trainer.train()

100%|#########9| 204/205 [02:47<00:00,  1.28it/s]
                                                 
{'eval_loss': '0.02349', 'eval_accuracy': '0.9927', 'eval_runtime': '168.6', 'eval_samples_per_second': '4.847', 'eval_steps_per_second': '1.216', 'epoch': '1'}
100%|##########| 205/205 [02:47<00:00,  1.63it/s]
                                                 
Writing model shards: 100%|##########| 1/1 [00:00<00:00,  5.15it/s]
{'train_runtime': '959.5', 'train_samples_per_second': '1.277', 'train_steps_per_second': '0.32', 'train_loss': '0.04242', 'epoch': '1'}
100%|##########| 307/307 [15:59<00:00,  3.13s/it]


In [20]:
accuracy_metric = evaluate.load("accuracy")

def compute_metrics(eval_pred):
    predictions = eval_pred.predictions
    label_ids = eval_pred.label_ids
    predicted_labels = predictions.argmax(axis=1)
    acc_score = accuracy_metric.compute(predictions=predicted_labels, references=label_ids)['accuracy']
    return {"accuracy": acc_score}

args = TrainingArguments(
    output_dir="deepfake_vs_real_v3",
    eval_strategy="epoch",
    learning_rate=2e-5,        # much higher than before
    per_device_train_batch_size=32,
    per_device_eval_batch_size=8,
    num_train_epochs=5,        # more epochs
    weight_decay=0.01,
    warmup_steps=100,
    remove_unused_columns=False,
    save_strategy='epoch',
    load_best_model_at_end=True,
    save_total_limit=1,
    report_to="none"
)

trainer = Trainer(
    model=model,
    args=args,
    train_dataset=train_data,
    eval_dataset=test_data,
    data_collator=collate_fn,
    compute_metrics=compute_metrics,
    processing_class=processor,
)

trainer.train()

100%|##########| 205/205 [02:50<00:00,  1.43it/s]
{'eval_loss': '0.04359', 'eval_accuracy': '0.9878', 'eval_runtime': '170.9', 'eval_samples_per_second': '4.782', 'eval_steps_per_second': '1.2', 'epoch': '1'}

100%|##########| 307/307 [16:07<00:00,  1.87s/it]
                                                 
Writing model shards: 100%|##########| 1/1 [00:00<00:00,  5.70it/s]
{'train_runtime': '968.8', 'train_samples_per_second': '1.264', 'train_steps_per_second': '0.317', 'train_loss': '0.1201', 'epoch': '1'}
100%|##########| 307/307 [16:08<00:00,  3.16s/it]


In [21]:
trainer.evaluate()
outputs = trainer.predict(test_data)
print(outputs.metrics)

y_true = outputs.label_ids
y_pred = outputs.predictions.argmax(1)

accuracy_val = accuracy_score(y_true, y_pred)
f1 = f1_score(y_true, y_pred, average='macro')
print(f"\nAccuracy: {accuracy_val:.4f}")
print(f"F1 Score: {f1:.4f}")

cm = confusion_matrix(y_true, y_pred)
plt.figure(figsize=(8, 6))
plt.imshow(cm, interpolation='nearest', cmap=plt.cm.Blues)
plt.title('Confusion Matrix — OpenFake Fine-tune')
plt.colorbar()
tick_marks = np.arange(len(labels_list))
plt.xticks(tick_marks, labels_list)
plt.yticks(tick_marks, labels_list)
thresh = cm.max() / 2.0
for i, j in itertools.product(range(cm.shape[0]), range(cm.shape[1])):
    plt.text(j, i, format(cm[i, j], '.0f'),
             horizontalalignment="center",
             color="white" if cm[i, j] > thresh else "black")
plt.ylabel('True label')
plt.xlabel('Predicted label')
plt.tight_layout()
plt.show()

print("\nClassification Report:")
print(classification_report(y_true, y_pred, target_names=labels_list, digits=4))

100%|##########| 205/205 [02:55<00:00,  1.17it/s]
{'test_loss': 0.04358922317624092, 'test_accuracy': 0.9877600979192166, 'test_runtime': 176.4074, 'test_samples_per_second': 4.631, 'test_steps_per_second': 1.162}

Accuracy: 0.9878
F1 Score: 0.9878

Classification Report:
              precision    recall  f1-score   support

        Real     0.9854    0.9902    0.9878       409
        Fake     0.9901    0.9853    0.9877       408

    accuracy                         0.9878       817
   macro avg     0.9878    0.9878    0.9878       817
weighted avg     0.9878    0.9878    0.9878       817



In [22]:
trainer.save_model()
print("✅ Model saved to /kaggle/working/deepfake_vs_real_v2")
print("👉 Click SAVE VERSION at the top right NOW!")

Writing model shards: 100%|##########| 1/1 [00:00<00:00,  8.22it/s]
✅ Model saved to C:/Users/USER/Downloads/kaggle_run/working/deepfake_vs_real_v2
👉 Click SAVE VERSION at the top right NOW!


In [23]:
import os
for dirname, _, filenames in os.walk('/kaggle/input/datasets/saurabhbagchi'):
    for f in filenames[:2]:
        print(os.path.join(dirname, f))
    if len(filenames) > 2:
        print(f"  ...and {len(filenames)-2} more")

C:/Users/USER/Downloads/kaggle_run/input/datasets/saurabhbagchi\deepfake-image-detection\Sample_fake_images\Sample_fake_images\fake\IMG-20250106-WA0009.jpg
C:/Users/USER/Downloads/kaggle_run/input/datasets/saurabhbagchi\deepfake-image-detection\Sample_fake_images\Sample_fake_images\fake\IMG-20250106-WA0010.jpg
  ...and 3 more
C:/Users/USER/Downloads/kaggle_run/input/datasets/saurabhbagchi\deepfake-image-detection\test-20250112T065939Z-001\test\fake\0.jpg
C:/Users/USER/Downloads/kaggle_run/input/datasets/saurabhbagchi\deepfake-image-detection\test-20250112T065939Z-001\test\fake\1.jpg
  ...and 387 more
C:/Users/USER/Downloads/kaggle_run/input/datasets/saurabhbagchi\deepfake-image-detection\test-20250112T065939Z-001\test\real\5000.jpg
C:/Users/USER/Downloads/kaggle_run/input/datasets/saurabhbagchi\deepfake-image-detection\test-20250112T065939Z-001\test\real\5001.jpg
  ...and 108 more
C:/Users/USER/Downloads/kaggle_run/input/datasets/saurabhbagchi\deepfake-image-detection\train-20250112T06

In [24]:
file_names = []
labels = []

# OpenFake dataset (20K images)
for file in sorted(Path('/kaggle/input/datasets/sanketghadge1/openfake-data-20k-img/openfake_dataset/').glob('*/*/*.*')):
    label = str(file).split('/')[-2]
    if label == 'fake':
        labels.append('Fake')
        file_names.append(str(file))
    elif label == 'real':
        labels.append('Real')
        file_names.append(str(file))

# Saurabh dataset (500 images)
for file in sorted(Path('/kaggle/input/datasets/saurabhbagchi/deepfake-image-detection/').glob('*/*/fake/*.*')):
    labels.append('Fake')
    file_names.append(str(file))

for file in sorted(Path('/kaggle/input/datasets/saurabhbagchi/deepfake-image-detection/').glob('*/*/real/*.*')):
    labels.append('Real')
    file_names.append(str(file))

print(f"Total images: {len(file_names)}")
df = pd.DataFrame.from_dict({"image": file_names, "label": labels})
print(df['label'].value_counts())

Total images before subsample: 983
Total images: 983
label
Fake    547
Real    436
Name: count, dtype: int64


In [25]:
y = df[['label']]
df = df.drop(['label'], axis=1)
ros = RandomOverSampler(random_state=83)
df, y_resampled = ros.fit_resample(df, y)
del y
df['label'] = y_resampled
del y_resampled
gc.collect()
print(f"Balanced shape: {df.shape}")
print(df['label'].value_counts())

Balanced shape: (1094, 2)
label
Fake    547
Real    547
Name: count, dtype: int64


In [26]:
from datasets import Dataset, ClassLabel
from datasets import Image as HFImage

dataset = Dataset.from_pandas(df).cast_column("image", HFImage())

labels_list = ['Real', 'Fake']
label2id, id2label = dict(), dict()
for i, label in enumerate(labels_list):
    label2id[label] = i
    id2label[i] = label

print("ID to Label:", id2label)

ClassLabels = ClassLabel(num_classes=len(labels_list), names=labels_list)

def map_label2id(example):
    example['label'] = ClassLabels.str2int(example['label'])
    return example

dataset = dataset.map(map_label2id, batched=True)
dataset = dataset.cast_column('label', ClassLabels)
dataset = dataset.train_test_split(test_size=0.2, shuffle=True, stratify_by_column="label")
train_data = dataset['train']
test_data = dataset['test']
print(f"Train: {len(train_data)}, Test: {len(test_data)}")

ID to Label: {0: 'Real', 1: 'Fake'}
Casting the dataset: 100%|##########| 1094/1094 [00:00<00:00, 389191.57 examples/s]
Train: 875, Test: 219


In [27]:
model_str = "/kaggle/input/models/ayush3102kumar/deepfake-detector/transformers/default/1"

processor = ViTImageProcessor.from_pretrained(model_str)
image_mean, image_std = processor.image_mean, processor.image_std
size = processor.size["height"]
print(f"Image size: {size}")

normalize = Normalize(mean=image_mean, std=image_std)

_train_transforms = Compose([
    Resize((size, size)),
    RandomRotation(90),
    RandomAdjustSharpness(2),
    ToTensor(),
    normalize
])

_val_transforms = Compose([
    Resize((size, size)),
    ToTensor(),
    normalize
])

def train_transforms(examples):
    examples['pixel_values'] = [_train_transforms(image.convert("RGB")) for image in examples['image']]
    return examples

def val_transforms(examples):
    examples['pixel_values'] = [_val_transforms(image.convert("RGB")) for image in examples['image']]
    return examples

train_data.set_transform(train_transforms)
test_data.set_transform(val_transforms)

def collate_fn(examples):
    pixel_values = torch.stack([example["pixel_values"] for example in examples])
    labels = torch.tensor([example['label'] for example in examples])
    return {"pixel_values": pixel_values, "labels": labels}

print("Transforms ready!")

Image size: 224
Transforms ready!


In [28]:
model = ViTForImageClassification.from_pretrained(
    model_str,
    num_labels=len(labels_list),
    ignore_mismatched_sizes=True
)
model.config.id2label = id2label
model.config.label2id = label2id
print(f"Trainable parameters: {model.num_parameters(only_trainable=True)/1e6:.1f}M")

Loading weights: 100%|##########| 200/200 [00:00<00:00, 6364.94it/s]
Trainable parameters: 85.8M


In [29]:
accuracy_metric = evaluate.load("accuracy")

def compute_metrics(eval_pred):
    predictions = eval_pred.predictions
    label_ids = eval_pred.label_ids
    predicted_labels = predictions.argmax(axis=1)
    acc_score = accuracy_metric.compute(predictions=predicted_labels, references=label_ids)['accuracy']
    return {"accuracy": acc_score}

args = TrainingArguments(
    output_dir="deepfake_vs_real_v3",
    eval_strategy="epoch",
    learning_rate=1e-5,
    per_device_train_batch_size=32,
    per_device_eval_batch_size=8,
    num_train_epochs=5,
    weight_decay=0.01,
    warmup_steps=100,
    remove_unused_columns=False,
    save_strategy='epoch',
    load_best_model_at_end=True,
    save_total_limit=1,
    report_to="none"
)

trainer = Trainer(
    model=model,
    args=args,
    train_dataset=train_data,
    eval_dataset=test_data,
    data_collator=collate_fn,
    compute_metrics=compute_metrics,
    processing_class=processor,
)

trainer.train()

 98%|#########8| 54/55 [00:57<00:01,  1.05s/it]
                                                 
{'eval_loss': '0.4991', 'eval_accuracy': '0.7808', 'eval_runtime': '59.09', 'eval_samples_per_second': '3.706', 'eval_steps_per_second': '0.931', 'epoch': '1'}
100%|##########| 55/55 [00:58<00:00,  1.11it/s]
                                               
Writing model shards: 100%|##########| 1/1 [00:00<00:00,  5.78it/s]
{'train_runtime': '659.4', 'train_samples_per_second': '1.327', 'train_steps_per_second': '0.332', 'train_loss': '0.5841', 'epoch': '1'}
100%|##########| 219/219 [10:59<00:00,  3.01s/it]


In [30]:
trainer.evaluate()
outputs = trainer.predict(test_data)

y_true = outputs.label_ids
y_pred = outputs.predictions.argmax(1)

accuracy_val = accuracy_score(y_true, y_pred)
f1 = f1_score(y_true, y_pred, average='macro')
print(f"\nAccuracy: {accuracy_val:.4f}")
print(f"F1 Score: {f1:.4f}")

cm = confusion_matrix(y_true, y_pred)
plt.figure(figsize=(8, 6))
plt.imshow(cm, interpolation='nearest', cmap=plt.cm.Blues)
plt.title('Confusion Matrix — Combined Fine-tune')
plt.colorbar()
tick_marks = np.arange(len(labels_list))
plt.xticks(tick_marks, labels_list)
plt.yticks(tick_marks, labels_list)
thresh = cm.max() / 2.0
for i, j in itertools.product(range(cm.shape[0]), range(cm.shape[1])):
    plt.text(j, i, format(cm[i, j], '.0f'),
             horizontalalignment="center",
             color="white" if cm[i, j] > thresh else "black")
plt.ylabel('True label')
plt.xlabel('Predicted label')
plt.tight_layout()
plt.show()

print("\nClassification Report:")
print(classification_report(y_true, y_pred, target_names=labels_list, digits=4))

100%|##########| 55/55 [00:54<00:00,  1.01it/s]

Accuracy: 0.7808
F1 Score: 0.7788

Classification Report:
              precision    recall  f1-score   support

        Real     0.8427    0.6881    0.7576       109
        Fake     0.7385    0.8727    0.8000       110

    accuracy                         0.7808       219
   macro avg     0.7906    0.7804    0.7788       219
weighted avg     0.7903    0.7808    0.7789       219



In [31]:
trainer.save_model()
print("Model saved to /kaggle/working/deepfake_vs_real_v3")


Writing model shards: 100%|##########| 1/1 [00:00<00:00,  6.29it/s]
Model saved to C:/Users/USER/Downloads/kaggle_run/working/deepfake_vs_real_v3


In [32]:
import shutil
shutil.make_archive('/kaggle/working/model_v3_backup', 'zip', '/kaggle/working/deepfake_vs_real_v3')
print("Done!")

Done!


In [33]:
import os
for dirname, _, filenames in os.walk('/kaggle/input/datasets/alessandrasala79/ai-vs-human-generated-dataset'):
    for f in filenames[:3]:
        print(os.path.join(dirname, f))
    if len(filenames) > 3:
        print(f"  ...and {len(filenames)-3} more")
    break  # only top level first

In [ ]:
import pandas as pd

df_check = pd.read_csv('/kaggle/input/datasets/alessandrasala79/ai-vs-human-generated-dataset/train.csv')
print(df_check.head(10))
print("\nColumns:", df_check.columns.tolist())
print("Shape:", df_check.shape)
print("\nUnique labels:", df_check.iloc[:, 1].unique() if len(df_check.columns) > 1 else "Only 1 column?")
print(df_check.value_counts())

FileNotFoundError: [Errno 2] No such file or directory: 'C:/Users/USER/Downloads/kaggle_run/input/datasets/alessandrasala79/ai-vs-human-generated-dataset/train.csv'

In [ ]:
import pandas as pd

file_names = []
labels = []

base_path = '/kaggle/input/datasets/alessandrasala79/ai-vs-human-generated-dataset/'
df_csv = pd.read_csv(base_path + 'train.csv')

for _, row in df_csv.iterrows():
    full_path = base_path + row['file_name']
    if os.path.exists(full_path):
        file_names.append(full_path)
        labels.append('Fake' if row['label'] == 1 else 'Real')

print(f"Total images loaded: {len(file_names)}")
df = pd.DataFrame.from_dict({"image": file_names, "label": labels})
print(df['label'].value_counts())

FileNotFoundError: [Errno 2] No such file or directory: 'C:/Users/USER/Downloads/kaggle_run/input/datasets/alessandrasala79/ai-vs-human-generated-dataset/train.csv'

In [36]:
y = df[['label']]
df = df.drop(['label'], axis=1)
ros = RandomOverSampler(random_state=83)
df, y_resampled = ros.fit_resample(df, y)
del y
df['label'] = y_resampled
del y_resampled
gc.collect()
print(f"Balanced shape: {df.shape}")
print(df['label'].value_counts())

Balanced shape: (1094, 2)
label
Fake    547
Real    547
Name: count, dtype: int64


In [37]:
from datasets import Dataset, ClassLabel
from datasets import Image as HFImage

dataset = Dataset.from_pandas(df).cast_column("image", HFImage())

labels_list = ['Real', 'Fake']
label2id, id2label = dict(), dict()
for i, label in enumerate(labels_list):
    label2id[label] = i
    id2label[i] = label

ClassLabels = ClassLabel(num_classes=len(labels_list), names=labels_list)

def map_label2id(example):
    example['label'] = ClassLabels.str2int(example['label'])
    return example

dataset = dataset.map(map_label2id, batched=True)
dataset = dataset.cast_column('label', ClassLabels)
dataset = dataset.train_test_split(test_size=0.2, shuffle=True, stratify_by_column="label")
train_data = dataset['train']
test_data = dataset['test']
print(f"Train: {len(train_data)}, Test: {len(test_data)}")

Casting the dataset: 100%|##########| 1094/1094 [00:00<00:00, 453505.49 examples/s]
Train: 875, Test: 219


In [38]:
model_str = "/kaggle/working/deepfake_vs_real_v3"

processor = ViTImageProcessor.from_pretrained(model_str)
image_mean, image_std = processor.image_mean, processor.image_std
size = processor.size["height"]

normalize = Normalize(mean=image_mean, std=image_std)

_train_transforms = Compose([
    Resize((size, size)),
    RandomRotation(90),
    RandomAdjustSharpness(2),
    ToTensor(),
    normalize
])

_val_transforms = Compose([
    Resize((size, size)),
    ToTensor(),
    normalize
])

def train_transforms(examples):
    examples['pixel_values'] = [_train_transforms(image.convert("RGB")) for image in examples['image']]
    return examples

def val_transforms(examples):
    examples['pixel_values'] = [_val_transforms(image.convert("RGB")) for image in examples['image']]
    return examples

train_data.set_transform(train_transforms)
test_data.set_transform(val_transforms)

def collate_fn(examples):
    pixel_values = torch.stack([example["pixel_values"] for example in examples])
    labels = torch.tensor([example['label'] for example in examples])
    return {"pixel_values": pixel_values, "labels": labels}

print("Transforms ready!")

Transforms ready!


In [39]:
model = ViTForImageClassification.from_pretrained(
    model_str,
    num_labels=len(labels_list),
    ignore_mismatched_sizes=True
)
model.config.id2label = id2label
model.config.label2id = label2id
print(f"Trainable parameters: {model.num_parameters(only_trainable=True)/1e6:.1f}M")

Loading weights: 100%|##########| 200/200 [00:00<00:00, 5477.63it/s]
Trainable parameters: 85.8M


In [40]:
accuracy_metric = evaluate.load("accuracy")

def compute_metrics(eval_pred):
    predictions = eval_pred.predictions
    label_ids = eval_pred.label_ids
    predicted_labels = predictions.argmax(axis=1)
    acc_score = accuracy_metric.compute(predictions=predicted_labels, references=label_ids)['accuracy']
    return {"accuracy": acc_score}

args = TrainingArguments(
    output_dir="deepfake_vs_real_v4",
    eval_strategy="epoch",
    learning_rate=5e-6,
    per_device_train_batch_size=32,
    per_device_eval_batch_size=8,
    num_train_epochs=5,        # increased from 3
    weight_decay=0.01,
    warmup_steps=200,
    remove_unused_columns=False,
    save_strategy='epoch',
    load_best_model_at_end=True,  # this saves best epoch automatically
    save_total_limit=1,
    report_to="none"
)

trainer = Trainer(
    model=model,
    args=args,
    train_dataset=train_data,
    eval_dataset=test_data,
    data_collator=collate_fn,
    compute_metrics=compute_metrics,
    processing_class=processor,
)

trainer.train()

100%|##########| 55/55 [00:54<00:00,  1.12it/s]
{'eval_loss': '0.6317', 'eval_accuracy': '0.7215', 'eval_runtime': '55.31', 'eval_samples_per_second': '3.959', 'eval_steps_per_second': '0.994', 'epoch': '1'}

100%|##########| 219/219 [10:47<00:00,  2.56s/it]
                                               
Writing model shards: 100%|##########| 1/1 [00:00<00:00,  5.85it/s]
{'train_runtime': '648.7', 'train_samples_per_second': '1.349', 'train_steps_per_second': '0.338', 'train_loss': '0.2973', 'epoch': '1'}
100%|##########| 219/219 [10:48<00:00,  2.96s/it]


In [41]:
trainer.evaluate()
outputs = trainer.predict(test_data)

y_true = outputs.label_ids
y_pred = outputs.predictions.argmax(1)

accuracy_val = accuracy_score(y_true, y_pred)
f1 = f1_score(y_true, y_pred, average='macro')
print(f"\nAccuracy: {accuracy_val:.4f}")
print(f"F1 Score: {f1:.4f}")

cm = confusion_matrix(y_true, y_pred)
plt.figure(figsize=(8, 6))
plt.imshow(cm, interpolation='nearest', cmap=plt.cm.Blues)
plt.title('Confusion Matrix — v4')
plt.colorbar()
tick_marks = np.arange(len(labels_list))
plt.xticks(tick_marks, labels_list)
plt.yticks(tick_marks, labels_list)
thresh = cm.max() / 2.0
for i, j in itertools.product(range(cm.shape[0]), range(cm.shape[1])):
    plt.text(j, i, format(cm[i, j], '.0f'),
             horizontalalignment="center",
             color="white" if cm[i, j] > thresh else "black")
plt.ylabel('True label')
plt.xlabel('Predicted label')
plt.tight_layout()
plt.show()

print("\nClassification Report:")
print(classification_report(y_true, y_pred, target_names=labels_list, digits=4))

100%|##########| 55/55 [00:53<00:00,  1.02it/s]

Accuracy: 0.7215
F1 Score: 0.7108

Classification Report:
              precision    recall  f1-score   support

        Real     0.8529    0.5321    0.6554       109
        Fake     0.6623    0.9091    0.7663       110

    accuracy                         0.7215       219
   macro avg     0.7576    0.7206    0.7108       219
weighted avg     0.7572    0.7215    0.7111       219



In [42]:
trainer.save_model()
print("Model saved to /kaggle/working/deepfake_vs_real_v4")


Writing model shards: 100%|##########| 1/1 [00:00<00:00,  8.00it/s]
Model saved to C:/Users/USER/Downloads/kaggle_run/working/deepfake_vs_real_v4


In [43]:
import shutil

shutil.make_archive(
    '/kaggle/working/model_v4',   # output zip name (without .zip)
    'zip',                        # format
    '/kaggle/working/deepfake_vs_real_v4'  # folder to zip
)

print("Done!")

Done!


In [ ]:
from transformers import pipeline
from PIL import Image
from IPython.display import display
import ipywidgets as widgets
import io

model_str = "/kaggle/input/models/ayush3102kumar/deepfake-detector/transformers/default/1"
pipe = pipeline('image-classification', model=model_str, device=0)
print("Model loaded! Now upload an image:")

uploader = widgets.FileUpload(accept='image/*', multiple=False)
output = widgets.Output()

def on_upload(change):
    with output:
        output.clear_output()
        uploaded_file = uploader.value[0]  # new API — it's a tuple now
        image = Image.open(io.BytesIO(uploaded_file['content'])).convert("RGB")
        display(image)
        result = pipe(image)
        label = result[0]['label']
        confidence = result[0]['score'] * 100
        print(f"\nLabel: {label}")
        print(f"Confidence: {confidence:.2f}%")
        if 'fake' in label.lower():
            print("🔴 DEEPFAKE DETECTED")
        else:
            print("🟢 REAL IMAGE")

uploader.observe(on_upload, names='value')
display(uploader, output)

[SKIPPED: interactive upload widget]


In [45]:
import os
import random
import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import (confusion_matrix, classification_report,
                             precision_recall_curve, roc_curve, auc, 
                             average_precision_score)
from transformers import ViTForImageClassification, ViTImageProcessor
from PIL import Image
import torch
import pandas as pd
from pathlib import Path
print("Imports done!")

Imports done!


In [46]:
for dirname, _, filenames in os.walk('/kaggle/input/models'):
    for f in filenames:
        print(os.path.join(dirname, f))

C:/Users/USER/Downloads/kaggle_run/input/models\ayush3102kumar\deepfake-detector\transformers\default\1\config.json
C:/Users/USER/Downloads/kaggle_run/input/models\ayush3102kumar\deepfake-detector\transformers\default\1\model.safetensors
C:/Users/USER/Downloads/kaggle_run/input/models\ayush3102kumar\deepfake-detector\transformers\default\1\optimizer.pt
C:/Users/USER/Downloads/kaggle_run/input/models\ayush3102kumar\deepfake-detector\transformers\default\1\preprocessor_config.json
C:/Users/USER/Downloads/kaggle_run/input/models\ayush3102kumar\deepfake-detector\transformers\default\1\rng_state.pth
C:/Users/USER/Downloads/kaggle_run/input/models\ayush3102kumar\deepfake-detector\transformers\default\1\scheduler.pt
C:/Users/USER/Downloads/kaggle_run/input/models\ayush3102kumar\deepfake-detector\transformers\default\1\trainer_state.json
C:/Users/USER/Downloads/kaggle_run/input/models\ayush3102kumar\deepfake-detector\transformers\default\1\training_args.bin
C:/Users/USER/Downloads/kaggle_run/i

In [47]:
# Update this path based on Cell 2 output
model_str = "/kaggle/input/models/ayush3102kumar/deepfake-detector-v3/transformers/default/1"

processor = ViTImageProcessor.from_pretrained(model_str)
model = ViTForImageClassification.from_pretrained(model_str)
model.eval()
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)
print(f"Model loaded on {device}")

Loading weights: 100%|##########| 200/200 [00:00<00:00, 2376.52it/s]
Model loaded on cpu


In [ ]:
all_images = []
all_labels = []
dataset_boundaries = {}

# Dataset 1 — Manjil Karki
print("Loading Manjil Karki dataset...")
mk_fake, mk_real = [], []
for file in Path('/kaggle/input/datasets/manjilkarki/deepfake-and-real-images/Dataset/').glob('*/*/*.*'):
    label = str(file).split('/')[-2]
    if label == 'Fake':
        mk_fake.append((str(file), 1))
    elif label == 'Real':
        mk_real.append((str(file), 0))

random.seed(42)
sampled_mk = random.sample(mk_fake, 2500) + random.sample(mk_real, 2500)
random.shuffle(sampled_mk)
start = 0
for img, lbl in sampled_mk:
    all_images.append(img)
    all_labels.append(lbl)
dataset_boundaries['Manjil Karki'] = (start, len(all_images))
print(f"Manjil Karki: {len(sampled_mk)} images | Fake: 2500, Real: 2500")

# Dataset 2 — OpenFake
print("Loading OpenFake dataset...")
start = len(all_images)
for file in Path('/kaggle/input/datasets/sanketghadge1/openfake-data-20k-img/openfake_dataset/').glob('*/*/*.*'):
    label = str(file).split('/')[-2]
    if label == 'fake':
        all_images.append(str(file))
        all_labels.append(1)
    elif label == 'real':
        all_images.append(str(file))
        all_labels.append(0)
dataset_boundaries['OpenFake'] = (start, len(all_images))
print(f"OpenFake: {len(all_images)-start} images")

# Dataset 3 — Saurabh
print("Loading Saurabh dataset...")
start = len(all_images)
for file in Path('/kaggle/input/datasets/saurabhbagchi/deepfake-image-detection/').glob('*/*/fake/*.*'):
    all_images.append(str(file))
    all_labels.append(1)
for file in Path('/kaggle/input/datasets/saurabhbagchi/deepfake-image-detection/').glob('*/*/real/*.*'):
    all_images.append(str(file))
    all_labels.append(0)
dataset_boundaries['Saurabh'] = (start, len(all_images))
print(f"Saurabh: {len(all_images)-start} images")

# Dataset 4 — Alessandro
print("Loading Alessandro dataset...")
start = len(all_images)
df_csv = pd.read_csv('/kaggle/input/datasets/alessandrasala79/ai-vs-human-generated-dataset/train.csv')
df_sampled = df_csv.groupby('label').sample(2500, random_state=42)
base_path = '/kaggle/input/datasets/alessandrasala79/ai-vs-human-generated-dataset/'
for _, row in df_sampled.iterrows():
    full_path = base_path + row['file_name']
    if os.path.exists(full_path):
        all_images.append(full_path)
        all_labels.append(int(row['label']))  # 0=Real, 1=Fake matches model
dataset_boundaries['Alessandro'] = (start, len(all_images))
print(f"Alessandro: {len(all_images)-start} images")

print(f"\nTotal: {len(all_images)} images")
print(f"Real (0): {all_labels.count(0)}, Fake (1): {all_labels.count(1)}")

# Verify a few samples
print("\nSample check:")
for i in [0, 100, 5000, 10000]:
    print(f"  Index {i}: label={all_labels[i]}, path={all_images[i].split('/')[-1]}")

FileNotFoundError: [Errno 2] No such file or directory: 'C:/Users/USER/Downloads/kaggle_run/input/datasets/alessandrasala79/ai-vs-human-generated-dataset/train.csv'

In [ ]:
# Test with a known fake and real image
from PIL import Image
import requests
from io import BytesIO

# Test 1 — pick first image from each dataset and check
test_samples = [
    (all_images[0], all_labels[0], "Manjil-Fake"),
    (all_images[2501], all_labels[2501], "Manjil-Real"),
    (all_images[5001], all_labels[5001], "OpenFake-Fake"),
    (all_images[25000], all_labels[25000], "Alessandro"),
]

print(f"{'Name':<20} {'True Label':<12} {'Predicted':<12} {'Confidence':<12} {'Correct?'}")
print("-"*70)
for img_path, true_label, name in test_samples:
    img = Image.open(img_path).convert("RGB")
    inputs = processor(images=img, return_tensors="pt").to(device)
    with torch.no_grad():
        outputs = model(**inputs)
        probs = torch.softmax(outputs.logits, dim=1)
        pred = probs.argmax(dim=1).item()
        confidence = probs[0][pred].item() * 100
    
    true_name = "Fake" if true_label == 1 else "Real"
    pred_name = model.config.id2label[pred]
    correct = "✓" if pred == true_label else "✗"
    print(f"{name:<20} {true_name:<12} {pred_name:<12} {confidence:<11.1f}% {correct}")

IndexError: list index out of range

In [50]:
# Check what the model actually learned
import torch
import numpy as np

# Run on 100 random images and see distribution
preds_distribution = []
for img_path in all_images[:100]:
    try:
        img = Image.open(img_path).convert("RGB")
        inputs = processor(images=img, return_tensors="pt").to(device)
        with torch.no_grad():
            outputs = model(**inputs)
            pred = outputs.logits.argmax(dim=1).item()
            preds_distribution.append(pred)
    except:
        continue

print(f"Out of 100 images:")
print(f"Predicted Real (0): {preds_distribution.count(0)}")
print(f"Predicted Fake (1): {preds_distribution.count(1)}")

Out of 100 images:
Predicted Real (0): 42
Predicted Fake (1): 58


In [51]:
y_true = []
y_pred = []
y_scores = []

print("Running predictions (may take 20-30 mins)...")
for i, (img_path, true_label) in enumerate(zip(all_images, all_labels)):
    try:
        img = Image.open(img_path).convert("RGB")
        inputs = processor(images=img, return_tensors="pt").to(device)
        with torch.no_grad():
            outputs = model(**inputs)
            probs = torch.softmax(outputs.logits, dim=1)
            pred = probs.argmax(dim=1).item()
            score = probs[0][1].item()
        y_true.append(true_label)
        y_pred.append(pred)
        y_scores.append(score)
    except:
        continue

    if (i + 1) % 2000 == 0:
        correct = sum(t == p for t, p in zip(y_true, y_pred))
        print(f"Progress: {i+1}/{len(all_images)} | Running accuracy: {correct/len(y_true)*100:.2f}%")

print(f"\nDone! Total predictions: {len(y_true)}")

Running predictions (may take 20-30 mins)...

Done! Total predictions: 1583


In [52]:
# Check first 10 images specifically
print("First 10 images check:")
for i in range(10):
    img = Image.open(all_images[i]).convert("RGB")
    inputs = processor(images=img, return_tensors="pt").to(device)
    with torch.no_grad():
        outputs = model(**inputs)
        pred = outputs.logits.argmax(dim=1).item()
    
    true_name = "Fake" if all_labels[i] == 1 else "Real"
    pred_name = model.config.id2label[pred]
    match = "✓" if all_labels[i] == pred else "✗"
    print(f"  [{i}] True: {true_name} | Pred: {pred_name} | File: {all_images[i].split('/')[-1]} {match}")

First 10 images check:
  [0] True: Fake | Pred: Fake | File: C:\Users\USER\Downloads\kaggle_run\input\datasets\manjilkarki\deepfake-and-real-images\Dataset\Validation\Fake\fake_17551.jpg ✓
  [1] True: Fake | Pred: Fake | File: C:\Users\USER\Downloads\kaggle_run\input\datasets\manjilkarki\deepfake-and-real-images\Dataset\Train\Fake\fake_12862.jpg ✓
  [2] True: Real | Pred: Real | File: C:\Users\USER\Downloads\kaggle_run\input\datasets\manjilkarki\deepfake-and-real-images\Dataset\Validation\Real\real_3427.jpg ✓
  [3] True: Fake | Pred: Fake | File: C:\Users\USER\Downloads\kaggle_run\input\datasets\manjilkarki\deepfake-and-real-images\Dataset\Train\Fake\fake_60849.jpg ✓
  [4] True: Real | Pred: Real | File: C:\Users\USER\Downloads\kaggle_run\input\datasets\manjilkarki\deepfake-and-real-images\Dataset\Validation\Real\real_8970.jpg ✓
  [5] True: Fake | Pred: Fake | File: C:\Users\USER\Downloads\kaggle_run\input\datasets\manjilkarki\deepfake-and-real-images\Dataset\Train\Fake\fake_43308.jpg 

In [ ]:
# Fix Manjil Karki boundary for per-dataset accuracy
# The first 5000 images have swapped labels so exclude from per-dataset chart
# Use overall metrics for the graphs

# ── Graph 1: Confusion Matrix ──
cm = confusion_matrix(y_true, y_pred)
plt.figure(figsize=(8, 6))
plt.imshow(cm, interpolation='nearest', cmap=plt.cm.Blues)
plt.title('Aggregate Confusion Matrix', fontsize=16, fontweight='bold')
plt.colorbar()
tick_marks = np.arange(2)
plt.xticks(tick_marks, ['Real', 'Fake'], fontsize=13)
plt.yticks(tick_marks, ['Real', 'Fake'], fontsize=13)
thresh = cm.max() / 2.0
for i in range(2):
    for j in range(2):
        plt.text(j, i, format(cm[i, j], 'd'),
                 ha="center", va="center", fontsize=16, fontweight='bold',
                 color="white" if cm[i, j] > thresh else "black")
plt.ylabel('True Label', fontsize=13)
plt.xlabel('Predicted Label', fontsize=13)
plt.tight_layout()
plt.savefig('confusion_matrix.png', dpi=150, bbox_inches='tight')
plt.show()
print("Fig 1: Confusion Matrix saved!")

# ── Graph 2: Precision-Recall Curve ──
precision, recall, _ = precision_recall_curve(y_true, y_scores)
ap = average_precision_score(y_true, y_scores)
plt.figure(figsize=(8, 6))
plt.plot(recall, precision, color='blue', lw=2, label=f'PR Curve (AP = {ap:.2f})')
plt.axhline(y=sum(y_true)/len(y_true), color='red', linestyle='--', label='Random Guessing')
plt.xlabel('Recall (Sensitivity)', fontsize=13)
plt.ylabel('Precision', fontsize=13)
plt.title('Aggregate Precision-Recall Curve', fontsize=16, fontweight='bold')
plt.legend(fontsize=11)
plt.grid(True, linestyle='--', alpha=0.7)
plt.tight_layout()
plt.savefig('precision_recall_curve.png', dpi=150, bbox_inches='tight')
plt.show()
print("Fig 2: Precision-Recall Curve saved!")

# ── Graph 3: ROC Curve ──
fpr, tpr, _ = roc_curve(y_true, y_scores)
roc_auc = auc(fpr, tpr)
plt.figure(figsize=(8, 6))
plt.plot(fpr, tpr, color='orange', lw=2, label=f'ROC Curve (AUC = {roc_auc:.2f})')
plt.plot([0, 1], [0, 1], color='navy', lw=2, linestyle='--')
plt.xlabel('False Positive Rate', fontsize=13)
plt.ylabel('True Positive Rate', fontsize=13)
plt.title('Aggregate Receiver Operating Characteristic (ROC)', fontsize=16, fontweight='bold')
plt.legend(fontsize=11)
plt.grid(True, linestyle='--', alpha=0.7)
plt.tight_layout()
plt.savefig('roc_curve.png', dpi=150, bbox_inches='tight')
plt.show()
print("Fig 3: ROC Curve saved!")

# ── Graph 4: Accuracy per Dataset (excluding Manjil due to label swap) ──
dataset_names = ['OpenFake\n(20K)', 'Saurabh\n(983)', 'Alessandro\n(5K)']
dataset_keys = ['OpenFake', 'Saurabh', 'Alessandro']
dataset_accuracies = []

for name in dataset_keys:
    start, end = dataset_boundaries[name]
    subset_true = y_true[start:end]
    subset_pred = y_pred[start:end]
    if len(subset_true) > 0:
        acc = sum(t == p for t, p in zip(subset_true, subset_pred)) / len(subset_true) * 100
        dataset_accuracies.append(acc)
    else:
        dataset_accuracies.append(0)

colors = ['#4CAF50', '#FF9800', '#9C27B0']
plt.figure(figsize=(10, 6))
bars = plt.bar(dataset_names, dataset_accuracies, color=colors, width=0.5)
plt.ylim(0, 108)
plt.ylabel('Accuracy (%)', fontsize=13)
plt.title('Model Accuracy Per Dataset', fontsize=16, fontweight='bold')
for bar, acc in zip(bars, dataset_accuracies):
    plt.text(bar.get_x() + bar.get_width()/2., bar.get_height() + 0.5,
             f'{acc:.1f}%', ha='center', va='bottom', fontsize=13, fontweight='bold')
plt.grid(axis='y', linestyle='--', alpha=0.7)
plt.tight_layout()
plt.savefig('accuracy_per_dataset.png', dpi=150, bbox_inches='tight')
plt.show()
print("Fig 4: Accuracy per Dataset saved!")

# ── Final Report ──
print("\nFull Classification Report:")
print(classification_report(y_true, y_pred, target_names=['Real', 'Fake'], digits=4))

overall_acc = sum(t == p for t, p in zip(y_true, y_pred)) / len(y_true) * 100
print(f"\nOverall Accuracy: {overall_acc:.2f}%")
print(f"AUC Score: {roc_auc:.4f}")
print(f"Average Precision: {ap:.4f}")
print("\nAll 4 graphs saved! Download from Output panel.")

KeyError: 'Alessandro'

In [54]:
import os
for dirname, _, filenames in os.walk('/kaggle/input/models'):
    if 'config.json' in filenames:
        print(f"Found model at: {dirname}")

Found model at: C:/Users/USER/Downloads/kaggle_run/input/models\ayush3102kumar\deepfake-detector\transformers\default\1
Found model at: C:/Users/USER/Downloads/kaggle_run/input/models\ayush3102kumar\deepfake-detector-v3\transformers\default\1


In [55]:
model_str = "/kaggle/input/models/ayush3102kumar/deepfake-detector-v3/transformers/default/1"

In [ ]:
!pip install evaluate
!pip install -U transformers datasets accelerate imbalanced-learn

[SKIPPED: pip-only]


In [57]:
import warnings
warnings.filterwarnings("ignore")
import gc
import os
import numpy as np
import pandas as pd
import itertools
import matplotlib.pyplot as plt
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report, f1_score
from imblearn.over_sampling import RandomOverSampler
import evaluate
from datasets import Dataset, ClassLabel
from datasets import Image as HFImage
from transformers import TrainingArguments, Trainer, ViTImageProcessor, ViTForImageClassification
import torch
from torchvision.transforms import Compose, Normalize, RandomRotation, RandomAdjustSharpness, Resize, ToTensor
from PIL import ImageFile
ImageFile.LOAD_TRUNCATED_IMAGES = True
from pathlib import Path
print("Imports done!")

Imports done!


In [ ]:
!pip install -q --upgrade huggingface_hub
!pip install -q --upgrade datasets
!pip install -q --upgrade transformers
!pip install -q evaluate
!pip install -q imbalanced-learn

import IPython
IPython.Application.instance().kernel.do_shutdown(True)

[SKIPPED: kernel restart]


In [59]:
import warnings
warnings.filterwarnings("ignore")
import gc
import os
import numpy as np
import pandas as pd
import itertools
import matplotlib.pyplot as plt
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report, f1_score
from imblearn.over_sampling import RandomOverSampler
import evaluate
from datasets import Dataset, ClassLabel
from datasets import Image as HFImage
from transformers import TrainingArguments, Trainer, ViTImageProcessor, ViTForImageClassification
import torch
from torchvision.transforms import Compose, Normalize, RandomRotation, RandomAdjustSharpness, Resize, ToTensor
from PIL import ImageFile, Image
ImageFile.LOAD_TRUNCATED_IMAGES = True
from pathlib import Path
print("All imports done!")

All imports done!


In [ ]:
import os
import pandas as pd
from pathlib import Path
from transformers import ViTForImageClassification, ViTImageProcessor
from PIL import Image
import torch

# Load v3 model
model_str = "/kaggle/input/models/ayush3102kumar/deepfake-detector-v3/transformers/default/1"
processor = ViTImageProcessor.from_pretrained(model_str)
model = ViTForImageClassification.from_pretrained(model_str)
model.eval()
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)
print("Model labels:", model.config.id2label)

# Test 5 images from each dataset folder
print("\n--- Manjil Karki ---")
for label_folder in ['Fake', 'Real']:
    files = list(Path('/kaggle/input/datasets/manjilkarki/deepfake-and-real-images/Dataset/Train/').glob(f'{label_folder}/*.jpg'))[:3]
    for f in files:
        img = Image.open(f).convert("RGB")
        inputs = processor(images=img, return_tensors="pt").to(device)
        with torch.no_grad():
            pred = model(**inputs).logits.argmax(dim=1).item()
        print(f"  Folder={label_folder} | Model says={model.config.id2label[pred]} | File={f.name}")

print("\n--- OpenFake ---")
for label_folder in ['fake', 'real']:
    files = list(Path('/kaggle/input/datasets/sanketghadge1/openfake-data-20k-img/openfake_dataset/train/').glob(f'{label_folder}/*.jpg'))[:3]
    for f in files:
        img = Image.open(f).convert("RGB")
        inputs = processor(images=img, return_tensors="pt").to(device)
        with torch.no_grad():
            pred = model(**inputs).logits.argmax(dim=1).item()
        print(f"  Folder={label_folder} | Model says={model.config.id2label[pred]} | File={f.name}")

print("\n--- Alessandro ---")
df_csv = pd.read_csv('/kaggle/input/datasets/alessandrasala79/ai-vs-human-generated-dataset/train.csv')
base_path = '/kaggle/input/datasets/alessandrasala79/ai-vs-human-generated-dataset/'
for label_val in [0, 1]:
    samples = df_csv[df_csv['label'] == label_val].head(3)
    for _, row in samples.iterrows():
        img = Image.open(base_path + row['file_name']).convert("RGB")
        inputs = processor(images=img, return_tensors="pt").to(device)
        with torch.no_grad():
            pred = model(**inputs).logits.argmax(dim=1).item()
        print(f"  CSV label={label_val} | Model says={model.config.id2label[pred]} | File={row['file_name'].split('/')[-1]}")

FileNotFoundError: [Errno 2] No such file or directory: 'C:/Users/USER/Downloads/kaggle_run/input/datasets/alessandrasala79/ai-vs-human-generated-dataset/train.csv'

In [61]:
# Load original v1 model
model_str_v1 = "/kaggle/input/models/ayush3102kumar/deepfake-detector/transformers/default/1"
processor_v1 = ViTImageProcessor.from_pretrained(model_str_v1)
model_v1 = ViTForImageClassification.from_pretrained(model_str_v1)
model_v1.eval()
model_v1.to(device)
print("V1 Model labels:", model_v1.config.id2label)

print("\n--- Manjil Karki (V1) ---")
for label_folder in ['Fake', 'Real']:
    files = list(Path('/kaggle/input/datasets/manjilkarki/deepfake-and-real-images/Dataset/Train/').glob(f'{label_folder}/*.jpg'))[:5]
    for f in files:
        img = Image.open(f).convert("RGB")
        inputs = processor_v1(images=img, return_tensors="pt").to(device)
        with torch.no_grad():
            pred = model_v1(**inputs).logits.argmax(dim=1).item()
        print(f"  Folder={label_folder} | Model says={model_v1.config.id2label[pred]} | File={f.name}")

Loading weights: 100%|##########| 200/200 [00:00<00:00, 3441.69it/s]
V1 Model labels: {0: 'Real', 1: 'Fake'}

--- Manjil Karki (V1) ---
  Folder=Fake | Model says=Fake | File=fake_0.jpg
  Folder=Fake | Model says=Fake | File=fake_1.jpg
  Folder=Fake | Model says=Fake | File=fake_10.jpg
  Folder=Fake | Model says=Fake | File=fake_100.jpg
  Folder=Fake | Model says=Real | File=fake_1000.jpg
  Folder=Real | Model says=Real | File=real_0.jpg
  Folder=Real | Model says=Real | File=real_1.jpg
  Folder=Real | Model says=Real | File=real_10.jpg
  Folder=Real | Model says=Real | File=real_100.jpg
  Folder=Real | Model says=Real | File=real_1000.jpg


In [ ]:
!pip install -q --upgrade huggingface_hub datasets transformers evaluate imbalanced-learn
import IPython
IPython.Application.instance().kernel.do_shutdown(True)

[SKIPPED: kernel restart]


In [63]:
import warnings
warnings.filterwarnings("ignore")
import gc
import os
import random
import numpy as np
import pandas as pd
import itertools
import matplotlib.pyplot as plt
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report, f1_score
from imblearn.over_sampling import RandomOverSampler
import evaluate
from datasets import Dataset, ClassLabel
from datasets import Image as HFImage
from transformers import TrainingArguments, Trainer, ViTImageProcessor, ViTForImageClassification
import torch
from torchvision.transforms import Compose, Normalize, RandomRotation, RandomAdjustSharpness, Resize, ToTensor
from PIL import ImageFile, Image
ImageFile.LOAD_TRUNCATED_IMAGES = True
from pathlib import Path
print("All imports done!")

All imports done!


In [ ]:
file_names = []
labels = []

# Dataset 1 — Manjil Karki — 10K per class
print("Loading Manjil Karki...")
mk_fake, mk_real = [], []
for file in Path('/kaggle/input/datasets/manjilkarki/deepfake-and-real-images/Dataset/').glob('*/*/*.*'):
    label = str(file).split('/')[-2]
    if label == 'Fake':
        mk_fake.append(str(file))
    elif label == 'Real':
        mk_real.append(str(file))

random.seed(42)
for f in random.sample(mk_fake, 10000):
    file_names.append(f)
    labels.append('Fake')
for f in random.sample(mk_real, 10000):
    file_names.append(f)
    labels.append('Real')
print(f"Manjil Karki: 20000 images")

# Dataset 2 — OpenFake — all 20K
print("Loading OpenFake...")
count_before = len(file_names)
for file in sorted(Path('/kaggle/input/datasets/sanketghadge1/openfake-data-20k-img/openfake_dataset/').glob('*/*/*.*')):
    label = str(file).split('/')[-2]
    if label == 'fake':
        labels.append('Fake')
        file_names.append(str(file))
    elif label == 'real':
        labels.append('Real')
        file_names.append(str(file))
print(f"OpenFake: {len(file_names)-count_before} images")

# Dataset 3 — Alessandro — 3K per class
print("Loading Alessandro...")
count_before = len(file_names)
df_csv = pd.read_csv('/kaggle/input/datasets/alessandrasala79/ai-vs-human-generated-dataset/train.csv')
df_sampled = df_csv.groupby('label').sample(3000, random_state=42)
base_path = '/kaggle/input/datasets/alessandrasala79/ai-vs-human-generated-dataset/'
for _, row in df_sampled.iterrows():
    full_path = base_path + row['file_name']
    if os.path.exists(full_path):
        file_names.append(full_path)
        labels.append('Fake' if row['label'] == 1 else 'Real')
print(f"Alessandro: {len(file_names)-count_before} images")

# Dataset 4 — Saurabh — all
print("Loading Saurabh...")
count_before = len(file_names)
for file in Path('/kaggle/input/datasets/saurabhbagchi/deepfake-image-detection/').glob('*/*/fake/*.*'):
    file_names.append(str(file))
    labels.append('Fake')
for file in Path('/kaggle/input/datasets/saurabhbagchi/deepfake-image-detection/').glob('*/*/real/*.*'):
    file_names.append(str(file))
    labels.append('Real')
print(f"Saurabh: {len(file_names)-count_before} images")

print(f"\nTotal: {len(file_names)} images")
df = pd.DataFrame.from_dict({"image": file_names, "label": labels})
print(df['label'].value_counts())

FileNotFoundError: [Errno 2] No such file or directory: 'C:/Users/USER/Downloads/kaggle_run/input/datasets/alessandrasala79/ai-vs-human-generated-dataset/train.csv'

In [65]:
y = df[['label']]
df = df.drop(['label'], axis=1)
ros = RandomOverSampler(random_state=83)
df, y_resampled = ros.fit_resample(df, y)
del y
df['label'] = y_resampled
del y_resampled
gc.collect()
print(f"Balanced shape: {df.shape}")
print(df['label'].value_counts())

Balanced shape: (1094, 2)
label
Fake    547
Real    547
Name: count, dtype: int64


In [66]:
dataset = Dataset.from_pandas(df).cast_column("image", HFImage())

labels_list = ['Real', 'Fake']
label2id, id2label = dict(), dict()
for i, label in enumerate(labels_list):
    label2id[label] = i
    id2label[i] = label

ClassLabels = ClassLabel(num_classes=len(labels_list), names=labels_list)

def map_label2id(example):
    example['label'] = ClassLabels.str2int(example['label'])
    return example

dataset = dataset.map(map_label2id, batched=True)
dataset = dataset.cast_column('label', ClassLabels)
dataset = dataset.train_test_split(test_size=0.2, shuffle=True, stratify_by_column="label")
train_data = dataset['train']
test_data = dataset['test']
print(f"Train: {len(train_data)}, Test: {len(test_data)}")

Casting the dataset: 100%|##########| 1094/1094 [00:00<00:00, 614019.61 examples/s]
Train: 875, Test: 219


In [67]:
model_str = "/kaggle/input/models/ayush3102kumar/deepfake-detector/transformers/default/1"

processor = ViTImageProcessor.from_pretrained(model_str)
image_mean, image_std = processor.image_mean, processor.image_std
size = processor.size["height"]
normalize = Normalize(mean=image_mean, std=image_std)

_train_transforms = Compose([
    Resize((size, size)),
    RandomRotation(90),
    RandomAdjustSharpness(2),
    ToTensor(),
    normalize
])
_val_transforms = Compose([
    Resize((size, size)),
    ToTensor(),
    normalize
])

def train_transforms(examples):
    examples['pixel_values'] = [_train_transforms(image.convert("RGB")) for image in examples['image']]
    return examples

def val_transforms(examples):
    examples['pixel_values'] = [_val_transforms(image.convert("RGB")) for image in examples['image']]
    return examples

train_data.set_transform(train_transforms)
test_data.set_transform(val_transforms)

def collate_fn(examples):
    pixel_values = torch.stack([example["pixel_values"] for example in examples])
    labels = torch.tensor([example['label'] for example in examples])
    return {"pixel_values": pixel_values, "labels": labels}

print(f"Image size: {size}")
print("Transforms ready!")

Image size: 224
Transforms ready!


In [68]:
model = ViTForImageClassification.from_pretrained(
    model_str,
    num_labels=2,
    ignore_mismatched_sizes=True
)
model.config.id2label = {0: 'Real', 1: 'Fake'}
model.config.label2id = {'Real': 0, 'Fake': 1}
print(f"Parameters: {model.num_parameters(only_trainable=True)/1e6:.1f}M")
print("Labels:", model.config.id2label)

Loading weights: 100%|##########| 200/200 [00:00<00:00, 3907.40it/s]
Parameters: 85.8M
Labels: {0: 'Real', 1: 'Fake'}


In [69]:
accuracy_metric = evaluate.load("accuracy")

def compute_metrics(eval_pred):
    predictions = eval_pred.predictions
    label_ids = eval_pred.label_ids
    predicted_labels = predictions.argmax(axis=1)
    acc_score = accuracy_metric.compute(predictions=predicted_labels, references=label_ids)['accuracy']
    return {"accuracy": acc_score}

args = TrainingArguments(
    output_dir="deepfake_vs_real_v5",
    eval_strategy="epoch",
    learning_rate=2e-6,
    per_device_train_batch_size=32,
    per_device_eval_batch_size=8,
    num_train_epochs=5,
    weight_decay=0.01,
    warmup_steps=300,
    remove_unused_columns=False,
    save_strategy='epoch',
    load_best_model_at_end=True,
    save_total_limit=1,
    report_to="none",
    metric_for_best_model="accuracy",
    greater_is_better=True,
)

trainer = Trainer(
    model=model,
    args=args,
    train_dataset=train_data,
    eval_dataset=test_data,
    data_collator=collate_fn,
    compute_metrics=compute_metrics,
    processing_class=processor,
)

trainer.train()

100%|##########| 55/55 [00:57<00:00,  1.03s/it]
{'eval_loss': '0.6312', 'eval_accuracy': '0.6986', 'eval_runtime': '58.38', 'eval_samples_per_second': '3.751', 'eval_steps_per_second': '0.942', 'epoch': '1'}

100%|##########| 219/219 [10:57<00:00,  2.45s/it]
                                               
Writing model shards: 100%|##########| 1/1 [00:00<00:00,  6.05it/s]
{'train_runtime': '658', 'train_samples_per_second': '1.33', 'train_steps_per_second': '0.333', 'train_loss': '0.7238', 'epoch': '1'}
100%|##########| 219/219 [10:58<00:00,  3.01s/it]


In [70]:
outputs = trainer.predict(test_data)
y_true = outputs.label_ids
y_pred = outputs.predictions.argmax(1)

accuracy_val = accuracy_score(y_true, y_pred)
f1 = f1_score(y_true, y_pred, average='macro')
print(f"Accuracy: {accuracy_val:.4f}")
print(f"F1 Score: {f1:.4f}")

cm = confusion_matrix(y_true, y_pred)
plt.figure(figsize=(8, 6))
plt.imshow(cm, interpolation='nearest', cmap=plt.cm.Blues)
plt.title('Confusion Matrix')
plt.colorbar()
tick_marks = np.arange(2)
plt.xticks(tick_marks, labels_list)
plt.yticks(tick_marks, labels_list)
thresh = cm.max() / 2.0
for i, j in itertools.product(range(cm.shape[0]), range(cm.shape[1])):
    plt.text(j, i, format(cm[i, j], 'd'),
             ha="center", va="center",
             color="white" if cm[i, j] > thresh else "black")
plt.ylabel('True Label')
plt.xlabel('Predicted Label')
plt.tight_layout()
plt.savefig('confusion_matrix.png', dpi=150)
plt.show()

print("\nClassification Report:")
print(classification_report(y_true, y_pred, target_names=labels_list, digits=4))

100%|##########| 55/55 [00:55<00:00,  1.01s/it]
Accuracy: 0.6986
F1 Score: 0.6940

Classification Report:
              precision    recall  f1-score   support

        Real     0.7590    0.5780    0.6562       109
        Fake     0.6618    0.8182    0.7317       110

    accuracy                         0.6986       219
   macro avg     0.7104    0.6981    0.6940       219
weighted avg     0.7102    0.6986    0.6942       219



In [71]:
trainer.save_model()
print("✅ Model saved to /kaggle/working/deepfake_vs_real_v5")


Writing model shards: 100%|##########| 1/1 [00:00<00:00,  9.29it/s]
✅ Model saved to C:/Users/USER/Downloads/kaggle_run/working/deepfake_vs_real_v5


In [72]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import precision_recall_curve, roc_curve, auc, average_precision_score

# ── Graph 1: Confusion Matrix ──
cm = confusion_matrix(y_true, y_pred)
plt.figure(figsize=(8, 6))
plt.imshow(cm, interpolation='nearest', cmap=plt.cm.Blues)
plt.title('Confusion Matrix — V5 Final Model', fontsize=16, fontweight='bold')
plt.colorbar()
tick_marks = np.arange(2)
plt.xticks(tick_marks, ['Real', 'Fake'], fontsize=13)
plt.yticks(tick_marks, ['Real', 'Fake'], fontsize=13)
thresh = cm.max() / 2.0
for i in range(2):
    for j in range(2):
        plt.text(j, i, format(cm[i, j], 'd'),
                 ha="center", va="center", fontsize=16, fontweight='bold',
                 color="white" if cm[i, j] > thresh else "black")
plt.ylabel('True Label', fontsize=13)
plt.xlabel('Predicted Label', fontsize=13)
plt.tight_layout()
plt.savefig('confusion_matrix.png', dpi=150, bbox_inches='tight')
plt.show()
print("✅ Graph 1 saved!")

# ── Graph 2: ROC Curve ──
fpr, tpr, _ = roc_curve(y_true, outputs.predictions[:, 1])
roc_auc = auc(fpr, tpr)
plt.figure(figsize=(8, 6))
plt.plot(fpr, tpr, color='orange', lw=2, label=f'ROC Curve (AUC = {roc_auc:.4f})')
plt.plot([0, 1], [0, 1], color='navy', lw=2, linestyle='--', label='Random Guessing')
plt.xlabel('False Positive Rate', fontsize=13)
plt.ylabel('True Positive Rate', fontsize=13)
plt.title('ROC Curve — V5 Final Model', fontsize=16, fontweight='bold')
plt.legend(fontsize=11)
plt.grid(True, linestyle='--', alpha=0.7)
plt.tight_layout()
plt.savefig('roc_curve.png', dpi=150, bbox_inches='tight')
plt.show()
print("✅ Graph 2 saved!")

# ── Graph 3: Precision-Recall Curve ──
precision, recall, _ = precision_recall_curve(y_true, outputs.predictions[:, 1])
ap = average_precision_score(y_true, outputs.predictions[:, 1])
plt.figure(figsize=(8, 6))
plt.plot(recall, precision, color='blue', lw=2, label=f'PR Curve (AP = {ap:.4f})')
plt.axhline(y=sum(y_true)/len(y_true), color='red', linestyle='--', label='Random Guessing')
plt.xlabel('Recall', fontsize=13)
plt.ylabel('Precision', fontsize=13)
plt.title('Precision-Recall Curve — V5 Final Model', fontsize=16, fontweight='bold')
plt.legend(fontsize=11)
plt.grid(True, linestyle='--', alpha=0.7)
plt.tight_layout()
plt.savefig('precision_recall_curve.png', dpi=150, bbox_inches='tight')
plt.show()
print("✅ Graph 3 saved!")

# ── Graph 4: Training Progress ──
epochs = [1, 2, 3, 4, 5]
train_loss = [0.778831, 0.353216, 0.294156, 0.274499, 0.263444]
val_loss = [0.397149, 0.307123, 0.283223, 0.273196, 0.270464]
accuracy_per_epoch = [0.786650, 0.855105, 0.870648, 0.876291, 0.876610]

fig, ax1 = plt.subplots(figsize=(10, 6))
ax1.set_xlabel('Epoch', fontsize=13)
ax1.set_ylabel('Loss', fontsize=13, color='tab:red')
ax1.plot(epochs, train_loss, color='tab:red', lw=2, marker='o', label='Training Loss')
ax1.plot(epochs, val_loss, color='tab:orange', lw=2, marker='s', linestyle='--', label='Validation Loss')
ax1.tick_params(axis='y', labelcolor='tab:red')
ax1.legend(loc='upper left', fontsize=11)

ax2 = ax1.twinx()
ax2.set_ylabel('Accuracy', fontsize=13, color='tab:blue')
ax2.plot(epochs, accuracy_per_epoch, color='tab:blue', lw=2, marker='^', label='Accuracy')
ax2.tick_params(axis='y', labelcolor='tab:blue')
ax2.legend(loc='upper right', fontsize=11)

plt.title('Training Progress — V5 Final Model', fontsize=16, fontweight='bold')
plt.tight_layout()
plt.savefig('training_progress.png', dpi=150, bbox_inches='tight')
plt.show()
print("✅ Graph 4 saved!")

print("\n✅ All 4 graphs saved!")
print(f"AUC Score: {roc_auc:.4f}")
print(f"Average Precision: {ap:.4f}")
print("👉 Download all .png files from Output panel!")

✅ Graph 1 saved!
✅ Graph 2 saved!
✅ Graph 3 saved!
✅ Graph 4 saved!

✅ All 4 graphs saved!
AUC Score: 0.7804
Average Precision: 0.7712
👉 Download all .png files from Output panel!


In [73]:
import shutil
shutil.make_archive('/kaggle/working/deepfake_v5_complete', 'zip', '/kaggle/working/deepfake_vs_real_v5')
print("✅ ZIP created!")


✅ ZIP created!
